#agent structure

1. **Database Access**
   1.1. The agent connects to an Oracle database using **SQLAlchemy**.
   1.2. Only **SELECT queries** are executed to ensure safety and prevent accidental modifications.

2. **Table Selection**
   2.1. An **orchestrator function** decides which table the agent should use for a user’s question.
   2.2. The function sends the question to the **LLM**, which selects the most appropriate table based on context.

3. **Table Context with Pydantic**
   3.1. Tables are represented as attributes in a **Pydantic class**, each corresponding to a database table.
   3.2. The `Field(description=...)` contains information about the table: its purpose, contained data, and what can be returned.
   3.3. These descriptions provide **context for the LLM**, helping it choose the correct table and reduce errors.

4. **Exploring the Selected Table**
   4.1. Once a table is chosen, the agent can perform a **table description (`get_desc`)** to inspect columns, data types, and other metadata.
   4.2. This helps the agent construct queries correctly and understand the table structure.

5. **Data Retrieval (Query Execution)**
   5.1. The agent builds **filtered and limited queries**, always prioritizing:
       - Relevant filters extracted from the user’s question to reduce returned data.
       - Row limits (`LIMIT` or Oracle equivalent) to avoid overloading the LLM’s context.
   5.2. This prevents excessive data consumption and keeps query results manageable.

6. **Execution Pipeline**
   6.1. User question → Table orchestrator chooses table → `get_desc` if necessary → `get_select` with filters and limits → LLM generates final response.
   6.2. The design is modular, safe, and scalable, making it easy to maintain and add new tables.

7. **Best Practices**
   7.1. Never return all rows from large tables.
   7.2. Keep table descriptions up-to-date for accurate LLM context.
   7.3. Validate filters before executing queries.
   7.4. Limit queries to **SELECT only** for security.

In [ ]:
#imports
from pydantic import BaseModel,Field, SecretStr
from pydantic_settings import BaseSettings
from dotenv import load_dotenv
from typing import Annotated, TypedDict